In [1]:
%%writefile app.py
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import streamlit as st

from gsec_data import build_table
from gsec_changes import all_changes
from gsec_labels import spreads_from_table, noise_floor, label_period

st.set_page_config(page_title="Indian G-Sec yield dashboard", layout="wide")


@st.cache_data
def load_all():
    table = build_table(str(Path(__file__).parent / "data"))
    daily, weekly, monthly = all_changes(table)
    spreads = spreads_from_table(table)
    return table, daily, weekly, monthly, spreads


table, daily, weekly, monthly, spreads = load_all()
thr = noise_floor(spreads, 10)


def fmt(d):
    return d.strftime("%d %b %Y")


st.title("Indian G-Sec yield dashboard")
st.caption("Yields are interpolated from individual G-Sec YTMs in FBIL's daily price files. "
           "Dates are sampled, not every business day.")

dates = list(table.index)
selected = st.sidebar.selectbox("Date", dates, index=len(dates) - 1, format_func=fmt)
earlier = [d for d in dates if d < selected]
if not earlier:
    st.warning("Pick a later date: there is no earlier date to compare with.")
    st.stop()
target = selected - pd.Timedelta(days=28)
default = min(range(len(earlier)), key=lambda i: abs((earlier[i] - target).days))
compare = st.sidebar.selectbox("Compare with", earlier, index=default, format_func=fmt)

c1, c2, c3 = st.columns(3)
y_now, y_then = table.loc[selected, 10], table.loc[compare, 10]
c1.metric("10Y yield", f"{y_now:.2f}%",
          f"{(y_now - y_then) * 100:+.1f} bps vs {fmt(compare)}", delta_color="off")
c2.metric("2s10s spread", f"{spreads.loc[selected, '2s10s']:.1f} bps",
          f"{spreads.loc[selected, '2s10s'] - spreads.loc[compare, '2s10s']:+.1f} bps", delta_color="off")
c3.metric("5s10s spread", f"{spreads.loc[selected, '5s10s']:.1f} bps",
          f"{spreads.loc[selected, '5s10s'] - spreads.loc[compare, '5s10s']:+.1f} bps", delta_color="off")

left, right = st.columns(2)

with left:
    st.subheader("Yield curve")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(table.columns, table.loc[compare], marker="o", label=fmt(compare))
    ax.plot(table.columns, table.loc[selected], marker="o", color="black",
            linewidth=2.2, label=fmt(selected))
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Yield (%)")
    ax.legend()
    st.pyplot(fig)

with right:
    st.subheader("10Y yield history (sampled dates)")
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(table.index, table[10], marker="o")
    ax2.axvline(selected, color="black", linestyle="--", linewidth=1)
    ax2.set_ylabel("10Y yield (%)")
    fig2.autofmt_xdate()
    st.pyplot(fig2)

st.subheader(f"Yield changes to {fmt(selected)} (bps)")
changes = pd.DataFrame({
    "daily": daily.loc[selected],
    "weekly": weekly.loc[selected],
    "monthly": monthly.loc[selected],
})
changes.index.name = "tenor (years)"
st.dataframe(changes.round(1))
st.caption("Blank means no earlier date was close enough to compare with. Daily needs the previous "
           "business day, weekly a date within 1 day of 7 days earlier, monthly within 3 days of one month earlier.")

st.subheader(f"Curve shape: {fmt(compare)} to {fmt(selected)}")
st.dataframe(label_period(spreads, compare, selected, thr), hide_index=True)
st.caption("Steepening or flattening needs a spread change at least as large as the typical one-day move "
           "over the last 10 business days: "
           + ", ".join(f"{k} {v:.1f} bps" for k, v in thr.items()) + ".")

with st.expander("Data and limitations"):
    st.markdown(
        "- Source: FBIL daily G-Sec price files, downloaded by hand. FBIL does not authorise commercial use "
        "without written permission, so this is a personal, non-commercial study.\n"
        "- Floating-rate bonds are excluded.\n"
        "- Yields at fixed tenors come from straight-line interpolation between neighbouring bonds, "
        "not from FBIL's own fitted curve.\n"
        "- The 1Y point is noisy and is not used in the spread labels.\n"
        "- Dates are sampled: daily for the last 10 business days, weekly and about monthly before that.\n"
        "- The label threshold comes from only 10 days of data, so it is a rough rule."
    )

Writing app.py


In [2]:
from gsec_data import build_table
t = build_table("data")
print(t.index[0].date(), t.index[-1].date())

2025-07-30 2026-10-16


In [3]:
%%writefile requirements.txt
pandas
numpy
matplotlib
streamlit
openpyxl
pytest

Writing requirements.txt


In [4]:
%%writefile README.md
# Indian G-Sec Yield Dashboard

Tracks Indian government bond yields across maturities, compares the yield curve between dates, and labels curve steepening and flattening. Built with Python, pandas and Streamlit.

![Dashboard](images/dashboard_default.png)

![Weekly view](images/dashboard_weekly.png)

## What it does
- Builds a yield table (1Y to 30Y) from FBIL's daily G-Sec price files
- Calculates daily, weekly and about-monthly changes in bps
- Calculates curve spreads (2s5s, 2s10s, 5s10s, 10s30s) and labels steepening, flattening and inversion
- Streamlit dashboard with a date picker and a comparison date

## Main findings (16 Oct 2026 vs 18 Sep 2026)
- The 10Y yield is 7.40%, up 31.4 bps
- 2s5s steepened by 8.1 bps. 5s10s flattened by 5.6 bps and 10s30s by 14.0 bps
- 2s10s changed by +2.6 bps, which is borderline and counted as little changed
- No spread was inverted on any sampled date
- See RESEARCH_NOTE.md for possible drivers and sources

## Method
- Fixed-rate G-Secs only. Floating-rate bonds are excluded
- Straight-line interpolation of bond YTMs to fixed tenors
- Label threshold: the median absolute one-day move of each spread over the last 10 business days

## Data
Source: FBIL (www.fbil.org.in), daily G-Sec price files, downloaded by hand. FBIL does not authorise commercial use without written permission, so the raw files are not in this repository. To reproduce, download the files from FBIL into a folder called `data`. See DATA_SOURCES.md.

## Limitations
- Dates are sampled (daily for 10 days, weekly, then roughly monthly), not every business day
- Interpolated yields differ from the yield on a single benchmark bond
- The 1Y point is noisy and is not used in the spread labels
- The label threshold comes from only 10 days of data
- The meaning of FBIL remarks such as "Input Point" and "YC" has not been verified

## Run it
```
pip install -r requirements.txt
python -m streamlit run app.py
```

## Tests
```
python -m pytest -v
```

Writing README.md


In [5]:
%%writefile RESEARCH_NOTE.md
# What may be behind the rise in Indian G-Sec yields, Sep to Oct 2026

## What the data show
Between 18 Sep and 16 Oct 2026 the 10Y yield rose 31.4 bps to 7.40%. The 2s5s spread steepened by 8.1 bps, while 5s10s and 10s30s flattened by 5.6 and 14.0 bps. The 2s10s spread changed by only 2.6 bps, which is within the noise. No spread inverted on any sampled date. Most of the 10Y rise (about 28.7 bps) had already happened by 9 Oct.

## Possible drivers
1. Rate hike: On 7 Oct 2026 the RBI raised the repo rate by 25 bps to 5.50%, its first hike since February 2023, and changed its stance from neutral to calibrated tightening [1]. In my data the 10Y rose about 4 bps that day and about 7 bps the next day.
2. Liquidity: The RBI also raised the minimum daily cash reserve requirement for banks from 90% to 99%, from the fortnight starting 16 Oct, and an analyst linked the 12 Oct jump in the 10Y yield to this liquidity withdrawal [2]. On 12 Oct my data show the 3Y up 13.2 bps, the 5Y up 12.9 bps and the 10Y up 4.9 bps, so the move was concentrated in the shorter and middle maturities. That fits the idea that tighter liquidity raised short-term funding costs, but it does not prove it.
3. Oil and inflation: Brent crude was above USD 100 a barrel and markets expected September inflation of about 5.4% [3].
4. Supply and fund selling: The government planned to borrow nearly INR 8 trillion through March [4], and mutual funds were cutting longer-dated government bonds, with net sales of INR 378 billion in July to September [3].
5. Global rates: The US Federal Reserve raised rates by 25 bps in September 2026, and other major central banks were also tightening [5].

These are possible drivers reported by analysts. This analysis does not prove that any of them caused the moves.

## What this analysis cannot show
- Dates are sampled and the yields are interpolated, so they differ from the yield on a single benchmark bond. A news report put the 10Y rise on 12 Oct at about 10 bps, against 4.9 bps in my data, possibly because a report quotes one bond while I interpolate across many.
- Two things happening together does not show that one caused the other.
- Most of the rise came before the 12 Oct liquidity measure, so that measure cannot explain the whole move.

## Sources
1. SCC Online, "RBI hikes repo rate to 5.50%; shifts policy stance to calibrated tightening", 9 Oct 2026. https://www.scconline.com/blog/post/2026/10/09/rbi-hikes-repo-rate-to-5-50-percent-october-2026/
2. ANI, "India's 10-year bond yield hits 7.40%; experts see pressure on credit growth", 12 Oct 2026. https://www.aninews.in/news/business/indias-10-year-bond-yield-hits-740-experts-see-pressure-on-credit-growth20261012175744/
3. Trading Economics, "India 10Y Yield Returns to 3-Year High", mid-Oct 2026 (confirm date on page). https://tradingeconomics.com/india/government-bond-yield/news/591417
4. Trading Economics, "India 10Y Yield Hits Over 2-Year High", early Oct 2026 (confirm date on page). https://tradingeconomics.com/india/government-bond-yield/news/588615
5. Adda247, "Bi-monthly Monetary Policy: RBI Raises Repo Rate by 25 bps After Over 3 Years", Oct 2026 (confirm date on page). https://currentaffairs.adda247.com/bi-monthly-monetary-policy-rbi-raises-repo-rate-by-25-bps-after-over-3-years/

Writing RESEARCH_NOTE.md
